# LLM and AI Compliance in Honolulu DPP

This executable **Jupyter Notebook simulation** demonstrates how an AI compliance engine verifies 2D CAD/BIM metadata against municipal zoning bylaws and building codes.

The code simulates the 4-step pipeline used in automated plan review systems:

1. **Rule Ingestion:** Translates municipal PDF text into machine-readable logic.
2. **CAD Plan Data:** Defines extracted drawing geometry (ADU dimensions, window areas).
3. **Deterministic Verification Engine:** Evaluates spatial conditions without relying on LLM arithmetic.
4. **LLM Citation Generator:** Generates a cited compliance report for municipal reviewers.


### Step 1: Inputting Plans & Local Code Knowledge

An architect uploads a **2D architectural PDF set** for a proposed 600 sq. ft. ADU.

Simultaneously, the platform's backend maintains a structured knowledge base of local **Honolulu Revised Ordinances (ROH Zoning)** alongside the **International Residential Code (IRC)**:

* **Zoning Constraint:** *Maximum rear-yard setback = 10 ft; Maximum height = 15 ft.*
* **Building Code Constraint:** *Bedroom egress window minimum clear opening width = 20 in; net clear opening area = 5.7 sq. ft.*


### Step 2: Multimodal Extraction & Automated Rules Engine

The AI tool doesn't just read the text; it combines **Computer Vision** (to read floor plans, elevations, and dimension strings) with an **LLM** (to interpret notes, room tags, and code clauses):

1. **Zoning Setback Check:** The AI measures the spatial distance between the lot boundary line and the outer ADU wall on the plan set.

   * **Measured:** $8.5\text{ ft}$ rear setback.
   * **Rule:** Minimum $10.0\text{ ft}$.
   * **Status:** **FAIL**
2. **Egress Window Check:** The AI reads the schedule on Sheet A-2 for Bedroom 1.

   * **Measured:** Opening dimensions $24\text{ in} \times 36\text{ in}$ ($6.0\text{ sq. ft.}$).
   * **Rule:** Minimum $5.7\text{ sq. ft.}$
   * **Status:** **PASS**


### Step 3: Generating the Cited Audit Report

Before the city reviewer even opens the file, the applicant receives an **Automated Screening Report**:

| **Item Checked**         | **Requirement**                    | **Drawing Measurement** | **Status** | **Code Citation / Flag**                                                                                      |
| ------------------------ | ---------------------------------- | ----------------------- | ---------- | ------------------------------------------------------------------------------------------------------------- |
| **Rear Setback**         | Min $10\text{ ft}$                 | $8.5\text{ ft}$         | ❌ **Fail** | **ROH §21-3.70-1 (Zoning):** Proposed ADU encroaches $1.5\text{ ft}$ into rear yard setback. *See Sheet A-1.* |
| **Max Building Height**  | Max $15\text{ ft}$                 | $14.2\text{ ft}$        | **Pass**   | **ROH §21-3.70-2:** Complies with height ceiling limit.                                                       |
| **Egress Window Area**   | Min $5.7\text{ sq ft}$             | $6.0\text{ sq ft}$      | **Pass**   | **2021 IRC §R310.2.1:** Window meets clear opening requirement.                                               |
| **Smoke Alarm Location** | Required inside & outside bedrooms | Missing in hallway      | ❌ **Fail** |                                                                                                               |


### Step 4: Human Review & Expedited Approval

Instead of going through **3 to 6 iterative review cycles** over several months, the applicant resolves the two flagged items (adjusting the setback and adding the smoke alarm) and submits a corrected plan.

When the application reaches the City Plan Examiner:

* The AI dashboard presents the pre-screened plan with all **Passed/Failed** tags pinned directly onto the PDF sheets.
* The examiner performs a **guided verification** in **10–15 minutes** (compared to the usual **1–2 hours** required for a fully manual check).
* The human examiner makes the final sign-off, maintaining complete regulatory accountability.


 ## 🚀 Example

### Step 1: Initialize Environment & Load Regulatory Knowledge Base

In [1]:
# -*- coding: utf-8 -*-
import json
import pandas as pd

# ==============================================================================
# STEP 1: DEFINE MUNICIPAL KNOWLEDGE BASE (ZONING & BUILDING CODES)
# Uses \u00a7 for Unicode safety and capitalized Python booleans (True)
# ==============================================================================
ZONING_KNOWLEDGE_BASE = {
    "ROH_21_3_70_1": {
        "title": "Accessory Dwelling Unit Rear Yard Setback",
        "code_source": "Revised Ordinances of Honolulu \u00a721-3.70-1",
        "category": "ZONING",
        "rule_type": "MINIMUM_VALUE",
        "target_field": "rear_setback_ft",
        "threshold": 10.0,
        "unit": "ft"
    },
    "ROH_21_3_70_2": {
        "title": "ADU Maximum Height Limit",
        "code_source": "Revised Ordinances of Honolulu \u00a721-3.70-2",
        "category": "ZONING",
        "rule_type": "MAXIMUM_VALUE",
        "target_field": "building_height_ft",
        "threshold": 15.0,
        "unit": "ft"
    },
    "IRC_R310_2_1": {
        "title": "Emergency Escape Egress Window Area",
        "code_source": "2021 International Residential Code (IRC) \u00a7R310.2.1",
        "category": "BUILDING_SAFETY",
        "rule_type": "MINIMUM_VALUE",
        "target_field": "bedroom_1_egress_area_sqft",
        "threshold": 5.7,
        "unit": "sq ft"
    },
    "IRC_R314_3": {
        "title": "Smoke Alarm Hallway Presence",
        "code_source": "2021 International Residential Code (IRC) \u00a7R314.3",
        "category": "BUILDING_SAFETY",
        "rule_type": "BOOLEAN_TRUE",
        "target_field": "hallway_smoke_alarm_present",
        "threshold": True,
        "unit": "boolean"
    }
}

# Optional: Demonstrate safe JSON file write/read without missing file errors
json_file_path = "zoning_rules.json"
with open(json_file_path, "w", encoding="utf-8") as f:
    json.dump(ZONING_KNOWLEDGE_BASE, f, indent=4, ensure_ascii=True)

with open(json_file_path, "r", encoding="utf-8") as f:
    loaded_rules_db = json.load(f)

print("✅ Municipal Knowledge Base loaded successfully.")

✅ Municipal Knowledge Base loaded successfully.


### Step 2: Define Extracted CAD/BIM Plan Data

In [2]:
# ==============================================================================
# STEP 2: SIMULATE EXTRACTED CAD/BIM PLAN DATA
# ==============================================================================
extracted_drawing_data = {
    "project_id": "PERMIT-2026-ADU-0892",
    "project_address": "1042 Oahu Ave, Honolulu, HI",
    "sheet_references": {
        "site_plan": "Sheet A-1",
        "floor_plan": "Sheet A-2"
    },
    "extracted_parameters": {
        "rear_setback_ft": 8.5,                   # Non-compliant (< 10.0 ft)
        "building_height_ft": 14.2,               # Compliant (< 15.0 ft)
        "bedroom_1_egress_area_sqft": 6.0,        # Compliant (> 5.7 sq ft)
        "hallway_smoke_alarm_present": False       # Non-compliant (Missing)
    }
}

print(f"📄 Extracted metadata for: {extracted_drawing_data['project_id']}")

📄 Extracted metadata for: PERMIT-2026-ADU-0892


### Step 3: Run Deterministic Verification Engine

In [3]:
# ==============================================================================
# STEP 3: DETERMINISTIC VERIFICATION ENGINE
# ==============================================================================
def run_compliance_audit(plan_data, rules_db):
    results = []
    params = plan_data["extracted_parameters"]
    sheets = plan_data["sheet_references"]
    
    for rule_id, rule in rules_db.items():
        field = rule["target_field"]
        value = params.get(field)
        status = "FAIL"
        
        # Exact arithmetic and logical checks
        if rule["rule_type"] == "MINIMUM_VALUE":
            if value is not None and value >= rule["threshold"]:
                status = "PASS"
        elif rule["rule_type"] == "MAXIMUM_VALUE":
            if value is not None and value <= rule["threshold"]:
                status = "PASS"
        elif rule["rule_type"] == "BOOLEAN_TRUE":
            if value is True:
                status = "PASS"
                
        sheet_ref = sheets["site_plan"] if rule["category"] == "ZONING" else sheets["floor_plan"]
        
        results.append({
            "Rule ID": rule_id,
            "Check Title": rule["title"],
            "Measured Value": f"{value} {rule['unit']}" if rule['unit'] != 'boolean' else str(value),
            "Requirement": f"{rule['rule_type'].replace('_', ' ')}: {rule['threshold']} {rule['unit']}",
            "Status": status,
            "Citation": rule["code_source"],
            "Sheet Reference": sheet_ref
        })
        
    return pd.DataFrame(results)

# Run verification engine
audit_df = run_compliance_audit(extracted_drawing_data, loaded_rules_db)

### Step 4: LLM Prompt Formatting & Citation Report Generation

In [4]:
# ==============================================================================
# STEP 4: LLM-STYLE REPORT GENERATOR
# ==============================================================================
def generate_examiner_report(df, plan_info):
    failed_items = df[df["Status"] == "FAIL"]
    passed_items = df[df["Status"] == "PASS"]
    
    report = []
    report.append(f"# PRESCREENING COMPLIANCE REPORT: {plan_info['project_id']}")
    report.append(f"**Address:** {plan_info['project_address']}\n")
    report.append(f"### SUMMARY: {len(passed_items)} Passed | {len(failed_items)} Action Items Required\n")
    
    if not failed_items.empty:
        report.append("## ❌ ACTION ITEMS REQUIRED (NON-COMPLIANT FEATURES)\n")
        for _, row in failed_items.iterrows():
            report.append(f"#### 🔴 {row['Check Title']}")
            report.append(f"- **Finding:** Measured value is **{row['Measured Value']}** ({row['Requirement']}).")
            report.append(f"- **Code Citation:** [{row['Citation']}]")
            report.append(f"- **Drawing Location:** {row['Sheet Reference']}")
            report.append(f"- **Action Required:** Revise drawing to meet minimum requirements before re-submittal.\n")
            
    report.append("## ✅ COMPLIANT ITEMS")
    for _, row in passed_items.iterrows():
        report.append(f"- **{row['Check Title']}:** {row['Measured Value']} (Verified against {row['Citation']})")
        
    return "\n".join(report)

# Print Markdown output directly in Jupyter
report_markdown = generate_examiner_report(audit_df, extracted_drawing_data)
print(report_markdown)

# PRESCREENING COMPLIANCE REPORT: PERMIT-2026-ADU-0892
**Address:** 1042 Oahu Ave, Honolulu, HI

### SUMMARY: 2 Passed | 2 Action Items Required

## ❌ ACTION ITEMS REQUIRED (NON-COMPLIANT FEATURES)

#### 🔴 Accessory Dwelling Unit Rear Yard Setback
- **Finding:** Measured value is **8.5 ft** (MINIMUM VALUE: 10.0 ft).
- **Code Citation:** [Revised Ordinances of Honolulu §21-3.70-1]
- **Drawing Location:** Sheet A-1
- **Action Required:** Revise drawing to meet minimum requirements before re-submittal.

#### 🔴 Smoke Alarm Hallway Presence
- **Finding:** Measured value is **False** (BOOLEAN TRUE: True boolean).
- **Code Citation:** [2021 International Residential Code (IRC) §R314.3]
- **Drawing Location:** Sheet A-2
- **Action Required:** Revise drawing to meet minimum requirements before re-submittal.

## ✅ COMPLIANT ITEMS
- **ADU Maximum Height Limit:** 14.2 ft (Verified against Revised Ordinances of Honolulu §21-3.70-2)
- **Emergency Escape Egress Window Area:** 6.0 sq ft (Verified agai